# 1 · Schema enrichment  (run once)

Builds the schema.org retrieval vocabulary that the matching notebook depends on. **One-time**:
re-run only to refresh the definitions — the per-article notebooks just load the JSON this writes.

Steps:
1. **Parse** `schemaorg-current-https.jsonld` into ~933 classes, each with its ancestor path.
2. **Enrich** every class comment with Gemini (batched, cached, interrupt-safe).
3. **Build** `schema_classes_enriched.json` — the enriched document pool (Examples / "Distinct
   from" clauses stripped from the embedded text, kept in `enriched_comment`).
4. *(appendix)* Export human-readable type / property trees to `schema_org_files/reference/`.

Outputs (all under `schema_org_files/`): `schema_classes.json`, `enriched_defs_cache.json`,
`schema_classes_enriched.json`, and the `reference/` exports.


In [ ]:
# --- Config -------------------------------------------------------------------------------
from pathlib import Path

SCHEMA_DIR = Path("schema_org_files")     # shared schema.org assets
REF_DIR    = SCHEMA_DIR / "reference"     # human-readable type/property exports
REF_DIR.mkdir(parents=True, exist_ok=True)

GEMINI_MODEL = "gemini-2.5-flash"         # authors the enriched definitions

# Enrichment knobs
ENRICH_BATCH = 100                        # schema types per Gemini call
MAX_TYPES    = None                       # set an int for a quick dry run; None = all ~933 types
GEMINI_DEFS_CACHE = SCHEMA_DIR / "enriched_defs_cache.json"   # label -> enriched def (resumable)
print("gemini:", GEMINI_MODEL, "| schema dir:", SCHEMA_DIR)

In [ ]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None

def gemini_json(system_prompt, user_message, model=None, temperature=0):
    """Call Gemini with JSON-mode output; return the parsed object and record latency."""
    global last_latency_s
    t0 = time.perf_counter()
    resp = client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            response_mime_type="application/json",
        ),
    )
    last_latency_s = time.perf_counter() - t0
    return json.loads(resp.text)

## 1. Parse schema.org into classes with hierarchy paths

In [ ]:
import re

def plain(value) -> str:
    if isinstance(value, dict):
        value = value.get("@value", "")
    value = re.sub(r"<[^>]+>", " ", value or "")
    return re.sub(r"\s+", " ", value).strip()

def as_list(x):
    if x is None: return []
    return x if isinstance(x, list) else [x]

raw = json.loads((SCHEMA_DIR / "schemaorg-current-https.jsonld").read_text())
classes = {}
for node in raw["@graph"]:
    if "rdfs:Class" not in as_list(node.get("@type")):
        continue
    c_id = node["@id"]
    if not c_id.startswith("schema:"):
        continue
    parents = [p["@id"] for p in as_list(node.get("rdfs:subClassOf"))
               if isinstance(p, dict) and p.get("@id", "").startswith("schema:")]
    classes[c_id] = {
        "id": c_id,
        "label": plain(node.get("rdfs:label")) or c_id.split(":")[-1],
        "comment": plain(node.get("rdfs:comment")),
        "parents": parents,
    }

def hierarchy_path(cid, _seen=None):
    _seen = _seen or set()
    if cid in _seen or cid not in classes:
        return []
    _seen.add(cid)
    parents = classes[cid]["parents"]
    if not parents:
        return []
    p = parents[0]
    return hierarchy_path(p, _seen) + [classes[p]["label"]] if p in classes else []

class_list = []
for c in classes.values():
    class_list.append({
        "id": c["id"], "label": c["label"], "comment": c["comment"],
        "path": " > ".join(hierarchy_path(c["id"])),
    })
print(f"{len(class_list)} schema.org classes parsed")

In [ ]:
# Persist the parsed classes: the baseline document pool AND a cache of the 1.5 MB JSON-LD parse
# (so the matching notebook can load this instead of re-parsing). Same "Label (path): comment"
# embedding_text as the enriched pool, so the two pools differ only in the comment text.
schema_classes = [
    {"id": c["id"], "label": c["label"], "comment": c["comment"], "path": c["path"],
     "embedding_text": (f"{c['label']} ({c['path']}): {c['comment']}" if c["path"]
                        else f"{c['label']}: {c['comment']}")}
    for c in class_list
]
(SCHEMA_DIR / "schema_classes.json").write_text(json.dumps(schema_classes, indent=2, ensure_ascii=False))
print(f"wrote {len(schema_classes)} classes -> schema_classes.json")

## 2. Enrich schema comments with Gemini

Batched and cached to `schema_org_files/enriched_defs_cache.json`. Interrupt-safe: re-running
skips types already in the cache, so you only pay for what's missing. Set `MAX_TYPES` in the
config cell for a quick partial run.

In [ ]:
ENRICH_SYSTEM_PROMPT = """You are building a retrieval vocabulary for the schema.org type system. You are given a BATCH of schema.org types, each with its ancestor path (root-first, e.g. "Thing > Place > AdministrativeArea") and its original schema.org comment. For EACH type, write one enriched definition that will later be embedded and matched against free-form entity-type labels.

Write each definition as a few compact sentences in THIS exact order and style:
1. A precise one-sentence gloss of what the type means. Use the ancestor path to pin down the intended sense (e.g. a "Table" under CreativeWork is a data table, not furniture).
2. "Also called <2-5 synonyms or alternative phrasings>."
3. "Examples: <3-6 concrete real-world instances>."
4. ONLY if the type is easily confused with a sibling type, add: "Distinct from <Sibling>, which is <one short clause>." Otherwise omit this sentence.

Keep every definition short and information-dense: long generic prose dilutes a mean-pooled embedding. Do not restate the label verbatim as the whole gloss. Stay faithful to schema.org's intended meaning.

Return ONLY valid JSON in this schema, covering EVERY label in the batch exactly once:
{ "definitions": { "<Label>": "<enriched definition>", "...": "..." } }"""

defs = json.loads(GEMINI_DEFS_CACHE.read_text()) if GEMINI_DEFS_CACHE.exists() else {}

pool = class_list if MAX_TYPES is None else class_list[:MAX_TYPES]
todo = [c for c in pool if c["label"] not in defs]
print(f"{len(defs)} cached | {len(todo)} to enrich | batch={ENRICH_BATCH}")

MAX_RETRIES = 5          # attempts per batch before giving up (transient errors: rate limits, timeouts)
t_start = time.perf_counter()
for i in range(0, len(todo), ENRICH_BATCH):
    batch = todo[i:i + ENRICH_BATCH]
    payload = [{"label": c["label"], "path": c["path"], "comment": c["comment"]} for c in batch]
    bnum = i // ENRICH_BATCH

    # Retry with exponential backoff; do NOT skip. If a batch can't be enriched after
    # MAX_RETRIES, persist what we have and stop hard so no types are silently dropped
    # (the cache lets you resume by simply re-running this cell).
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            out = gemini_json(ENRICH_SYSTEM_PROMPT, json.dumps(payload, ensure_ascii=False))
            got = out.get("definitions", {})
            break
        except Exception as e:
            if attempt == MAX_RETRIES:
                GEMINI_DEFS_CACHE.write_text(json.dumps(defs, indent=2, ensure_ascii=False))
                raise RuntimeError(
                    f"batch {bnum} failed after {MAX_RETRIES} attempts: {e}. "
                    f"{len(defs)} defs cached — re-run this cell to resume from here."
                ) from e
            wait = min(2 ** attempt, 30)
            print(f"  batch {bnum} attempt {attempt}/{MAX_RETRIES} failed ({e}); retrying in {wait}s ...")
            time.sleep(wait)

    # keep only non-empty defs for labels we actually asked about
    missing = []
    for c in batch:
        d = got.get(c["label"])
        if d and d.strip():
            defs[c["label"]] = d.strip()
        else:
            missing.append(c["label"])
    if missing:
        print(f"  batch {bnum}: model omitted {len(missing)} label(s), will retry next run: {missing[:8]}")
    GEMINI_DEFS_CACHE.write_text(json.dumps(defs, indent=2, ensure_ascii=False))
    done = min(i + ENRICH_BATCH, len(todo))
    print(f"  {len(defs)} total enriched  (+{done}/{len(todo)} this run, {last_latency_s:.1f}s/call)")

remaining = [c["label"] for c in pool if c["label"] not in defs]
print(f"done in {time.perf_counter()-t_start:.0f}s | {len(defs)}/{len(pool)} labels have a Gemini definition")
if remaining:
    print(f"{len(remaining)} still missing (model omitted them from its JSON) — re-run this cell to fill them: {remaining[:12]}")

## 3. Build the enriched document pool

In [ ]:
# Build the enriched document pool. Strip the "Examples: ..." sentence and contrastive
# drop the "Examples: ..." sentence and contrastive "Distinct from ..." clauses from the
# embedded text (they hurt a bi-encoder), but keep the full text in enriched_comment.
_CONTRAST_RE = re.compile(r"\s*(?:Distinct from|Not to be confused with)[^.]*\.", re.IGNORECASE)
_EXAMPLE_RE  = re.compile(r"\s*Examples?:.*?\.(?=\s+[A-Z]|\s*$)", re.IGNORECASE)

def embedding_comment(text):
    text = _CONTRAST_RE.sub("", text)
    text = _EXAMPLE_RE.sub("", text)
    return text.strip()

enriched_gemini = []
for c in class_list:
    ec = defs.get(c["label"])
    is_en = ec is not None
    comment_for_embed = embedding_comment(ec) if is_en else c["comment"]
    head = f"{c['label']} ({c['path']})" if c["path"] else c["label"]
    enriched_gemini.append({
        "id": c["id"], "label": c["label"], "comment": c["comment"],
        "enriched_comment": ec or "", "enriched": is_en, "path": c["path"],
        "embedding_text": f"{head}: {comment_for_embed}",
    })

out_pool = SCHEMA_DIR / "schema_classes_enriched.json"
out_pool.write_text(json.dumps(enriched_gemini, indent=2, ensure_ascii=False))
print(f"{sum(c['enriched'] for c in enriched_gemini)}/{len(enriched_gemini)} enriched -> {out_pool}")
# spot check
for lbl in ["Person", "Country", "City"]:
    print("  ", next(c["embedding_text"] for c in enriched_gemini if c["label"] == lbl))

## Appendix · Reference exports (optional)

Human-readable trees of the schema.org **type** and **property** hierarchies, plus per-domain
property tables, written to `schema_org_files/reference/`. Handy for eyeballing the vocabulary or
for prompt context; not consumed by the matching notebook.

In [ ]:
import json
from collections import defaultdict

with open(SCHEMA_DIR / "schemaorg-current-https.jsonld") as f:
    data = json.load(f)


def as_list(x):
    """Normalise a field that may be missing, a single value, or a list."""
    if x is None:
        return []
    return x if isinstance(x, list) else [x]


def short(ref):
    """Turn {'@id': 'schema:MediaObject'} (or 'schema:MediaObject') into 'MediaObject'."""
    if isinstance(ref, dict):
        ref = ref.get("@id", "")
    return ref.split(":")[-1]


# Collect every class and its parent class(es) via rdfs:subClassOf
children = defaultdict(list)   # parent -> [child, child, ...]
parents = defaultdict(list)    # child  -> [parent, ...]
types = []

for item in data["@graph"]:
    item_type = item.get("@type")
    is_class = item_type == "rdfs:Class" or (
        isinstance(item_type, list) and "rdfs:Class" in item_type
    )
    if not is_class:
        continue

    name = item.get("rdfs:label")
    if isinstance(name, dict):
        name = name.get("@value")
    if isinstance(name, list):
        name = name[0]
    if not name:
        continue

    types.append(name)
    for parent_ref in as_list(item.get("rdfs:subClassOf")):
        parent = short(parent_ref)
        if parent:
            children[parent].append(name)
            parents[name].append(parent)

# Roots = classes that have no parent (e.g. "Thing")
roots = sorted(t for t in types if not parents.get(t))

print(f"Found {len(types)} types | {len(roots)} root(s): {roots}")

In [ ]:
# Save as an indented tree for entity typing: shows granularity, each type once, token-light.
# Place each class under its first parent that is itself a defined type (a few schema.org
# parents like "Document"/"LegalDocument" are referenced but not defined here, so we skip them).
typeset = set(types)
primary_children = defaultdict(list)
extra_roots = []
for t in types:
    parent = next((p for p in parents.get(t, []) if p in typeset), None)
    if parent:
        primary_children[parent].append(t)
    elif t not in roots:
        extra_roots.append(t)   # parent(s) undefined in this file -> treat as its own root


def walk(node, depth, seen, lines):
    lines.append("  " * depth + node)          # 2 spaces per level
    if node in seen:
        return
    seen.add(node)
    for child in sorted(set(primary_children.get(node, []))):
        walk(child, depth + 1, seen, lines)


lines, seen = [], set()
for root in roots + sorted(extra_roots):
    walk(root, 0, seen, lines)

with open(REF_DIR / "types.txt", "w") as f:
    f.write("\n".join(lines) + "\n")

assert len(seen) == len(types), f"dropped {len(types) - len(seen)} types!"
print(f"Wrote {len(lines)} lines covering all {len(types)} types")
print("\n".join(lines[:25]))

In [ ]:
# Extract all schema.org properties with subject (domain) and object (range) type constraints.
# schema.org uses schema:domainIncludes / schema:rangeIncludes (not rdfs:domain/range),
# meaning a property can apply to multiple domain types and accept multiple range types.

import json
from collections import defaultdict

with open(SCHEMA_DIR / "schemaorg-current-https.jsonld") as f:
    data = json.load(f)

def as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

def short(ref):
    if isinstance(ref, dict):
        ref = ref.get("@id", "")
    return ref.split(":")[-1]

properties = []
by_domain = defaultdict(list)   # domain_type -> [{property, range}]

for item in data["@graph"]:
    item_type = item.get("@type")
    is_property = item_type == "rdf:Property" or (
        isinstance(item_type, list) and "rdf:Property" in item_type
    )
    if not is_property:
        continue

    name = item.get("rdfs:label")
    if isinstance(name, dict):
        name = name.get("@value")
    if isinstance(name, list):
        name = name[0]
    if not name:
        continue

    domain = sorted({short(d) for d in as_list(item.get("schema:domainIncludes"))})
    range_ = sorted({short(r) for r in as_list(item.get("schema:rangeIncludes"))})

    entry = {"property": name, "domain": domain, "range": range_}
    properties.append(entry)
    for d in domain:
        by_domain[d].append({"property": name, "range": range_})

properties.sort(key=lambda p: p["property"])

# --- flat JSON (all properties) ---
with open(REF_DIR / "properties.json", "w") as f:
    json.dump(properties, f, indent=2, ensure_ascii=False)

# --- flat text for prompt use ---
# Format: propertyName | domain: A, B | range: X, Y
prop_lines = []
for p in properties:
    d_str = ", ".join(p["domain"]) if p["domain"] else "ANY"
    r_str = ", ".join(p["range"])  if p["range"]  else "ANY"
    prop_lines.append(f"{p['property']} | domain: {d_str} | range: {r_str}")

with open(REF_DIR / "properties.txt", "w") as f:
    f.write("\n".join(prop_lines) + "\n")

# --- grouped by domain type (useful when you know the subject type) ---
grouped = {dtype: sorted(props, key=lambda x: x["property"])
           for dtype, props in sorted(by_domain.items())}
with open(REF_DIR / "properties_by_domain.json", "w") as f:
    json.dump(grouped, f, indent=2, ensure_ascii=False)

print(f"Found {len(properties)} properties across {len(by_domain)} domain types")
print(f"Wrote: properties.json  |  properties.txt  |  properties_by_domain.json")
print("\nSample (first 8 properties):")
for p in properties[:8]:
    d_str = ", ".join(p["domain"]) if p["domain"] else "ANY"
    r_str = ", ".join(p["range"])  if p["range"]  else "ANY"
    print(f"  {p['property']:30s}  domain: {d_str:40s}  range: {r_str}")

In [ ]:
# Build an indented property hierarchy using rdfs:subPropertyOf,
# mirroring the type hierarchy in types.txt.

import json
from collections import defaultdict

with open(SCHEMA_DIR / "schemaorg-current-https.jsonld") as f:
    data = json.load(f)

def as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

def short(ref):
    if isinstance(ref, dict):
        ref = ref.get("@id", "")
    return ref.split(":")[-1]

prop_children = defaultdict(list)   # parent -> [child, ...]
prop_parents  = defaultdict(list)   # child  -> [parent, ...]
prop_names = []

for item in data["@graph"]:
    item_type = item.get("@type")
    is_property = item_type == "rdf:Property" or (
        isinstance(item_type, list) and "rdf:Property" in item_type
    )
    if not is_property:
        continue

    name = item.get("rdfs:label")
    if isinstance(name, dict):
        name = name.get("@value")
    if isinstance(name, list):
        name = name[0]
    if not name:
        continue

    prop_names.append(name)
    for parent_ref in as_list(item.get("rdfs:subPropertyOf")):
        parent = short(parent_ref)
        if parent:
            prop_children[parent].append(name)
            prop_parents[name].append(parent)

prop_set = set(prop_names)

# Roots = properties with no parent defined in the file
prop_roots = sorted(p for p in prop_names if not prop_parents.get(p))

# Properties whose listed parent isn't itself in the file → treat as extra roots
extra_prop_roots = []
primary_prop_children = defaultdict(list)
for p in prop_names:
    parent = next((par for par in prop_parents.get(p, []) if par in prop_set), None)
    if parent:
        primary_prop_children[parent].append(p)
    elif p not in prop_roots:
        extra_prop_roots.append(p)

def walk_props(node, depth, seen, lines):
    lines.append("  " * depth + node)
    if node in seen:
        return
    seen.add(node)
    for child in sorted(set(primary_prop_children.get(node, []))):
        walk_props(child, depth + 1, seen, lines)

lines, seen = [], set()
for root in prop_roots + sorted(extra_prop_roots):
    walk_props(root, 0, seen, lines)

with open(REF_DIR / "property_hierarchy.txt", "w") as f:
    f.write("\n".join(lines) + "\n")

assert len(seen) == len(prop_names), f"dropped {len(prop_names) - len(seen)} properties!"
print(f"Found {len(prop_names)} properties | {len(prop_roots)} root(s)")
print(f"Wrote property_hierarchy.txt ({len(lines)} lines)")
print("\nFirst 20 lines:")
print("\n".join(lines[:20]))